# Clase 7 — Análisis exploratorio de datos (EDA)
**Taller de Ciencia de Datos (EIN146A25) · 2026-2 · Jueves 24 de septiembre**

Cuatro ejercicios sobre un mini-caso de **estudiantes de enseñanza media de la provincia de Concepción**. Al final, la plantilla del notebook `03-eda.ipynb` que deben aplicar a su propio proyecto hoy mismo.

**Datos sintéticos generados para el lab: ninguna cifra describe a una comuna, establecimiento o estudiante real.**

| # | Ejercicio | Qué se practica | Min |
|---|---|---|---|
| 1 | Una variable a la vez | histograma, escala log, `skew()`, nulo disfrazado | 20 |
| 2 | Relaciones y correlación | hexbin/muestreo, Pearson vs. Spearman, boxplot | 20 |
| 3 | Paradoja de Simpson | tabla por grupo, composición, segmentar con `hue` | 15 |
| 4 | Una figura que comunica | título = hallazgo, unidades, `savefig` | 10 |

Reglas del notebook:
- Los archivos de práctica **se generan solos** en `data_practica/` — no necesitan descargar nada.
- Las celdas marcadas **✏️ TU TURNO** son las que deben completar.
- Cada ejercicio trae su **verificación automática** (debe imprimir todos los ✓) y una **solución propuesta** plegada: intenten primero sin mirarla.
- Si un ejercicio no les sale, el siguiente puede usar la tabla de referencia `df_solN` que deja la celda del profesor.
- Ver `lab07-guia-eda.md` para la referencia técnica de cada sección (§N).

In [ ]:
# Setup — ejecutar primero
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

try:
    import seaborn as sns
    HAY_SNS = True
except ImportError:
    HAY_SNS = False
    print("seaborn no está instalado: todo el lab funciona igual solo con Matplotlib/pandas.")

CARPETA = Path("data_practica")
CARPETA.mkdir(exist_ok=True)
FIGS = Path("figs")


def spearman(a, b):
    '''Spearman = Pearson de los rangos, sobre las filas sin NaN en ninguna de las dos series.
    Evita depender de scipy (Series.corr(method="spearman") lo importa por dentro).'''
    m = a.notna() & b.notna()
    return a[m].rank().corr(b[m].rank())


def guardar_figura(fig, nombre):
    '''Guarda una figura en figs/ (la crea si no existe) con la resolución del informe.'''
    FIGS.mkdir(exist_ok=True)
    ruta = FIGS / nombre
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    print("Guardado:", ruta)


def ejecutar_checks(checks):
    '''checks: lista de (nombre, funcion_booleana_sin_argumentos).
    Atrapa NameError para que "Restart & Run All" con TU TURNO vacíos no se caiga.'''
    ok_total = True
    for nombre, chk in checks:
        try:
            ok = bool(chk())
        except NameError as e:
            var = str(e).split("'")[1]
            print(f"✗ {nombre} (todavía no definen `{var}`)")
            ok_total = False
            continue
        except Exception as e:
            print(f"✗ {nombre} (error al evaluar: {e})")
            ok_total = False
            continue
        print(("✓" if ok else "✗"), nombre)
        ok_total = ok_total and ok
    print("Todo OK" if ok_total else "Faltan checks")
    return ok_total


print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "· seaborn disponible:", HAY_SNS)
print("Carpeta de trabajo:", CARPETA.resolve())

In [ ]:
# Genera los archivos de práctica (no modificar esta celda)
rng = np.random.default_rng(2026)

COMUNAS = [("08101", "Concepción"), ("08102", "Coronel"), ("08103", "Chiguayante"),
           ("08104", "Florida"), ("08105", "Hualqui"), ("08106", "Lota"), ("08107", "Penco"),
           ("08108", "San Pedro de la Paz"), ("08109", "Santa Juana"), ("08110", "Talcahuano"),
           ("08111", "Tomé"), ("08112", "Hualpén")]
DEPS = ["Pública", "Part. subvencionado", "Part. pagado"]
FIJAS = {("Penco", "Pública"): (200, 40), ("Penco", "Part. subvencionado"): (1000, 110),
         ("Penco", "Part. pagado"): (800, 40), ("Tomé", "Pública"): (1600, 272),
         ("Tomé", "Part. subvencionado"): (340, 34), ("Tomé", "Part. pagado"): (60, 2)}
BASE = {"Pública": 0.16, "Part. subvencionado": 0.09, "Part. pagado": 0.04}

partes = []
per = (24000 - 4000) // 10
for cod, nom in COMUNAS:
    if nom in ("Penco", "Tomé"):
        for d in DEPS:
            n, k = FIJAS[(nom, d)]
            r = np.zeros(n, bool); r[:k] = True; rng.shuffle(r)
            partes.append(pd.DataFrame({"cod_comuna": cod, "comuna": nom, "dependencia": d, "riesgo": r}))
    else:
        comp = rng.dirichlet([4, 4, 1.5])
        ds = rng.choice(DEPS, size=per, p=comp)
        r = rng.random(per) < np.array([BASE[x] for x in ds])
        partes.append(pd.DataFrame({"cod_comuna": cod, "comuna": nom, "dependencia": ds, "riesgo": r}))
df = pd.concat(partes, ignore_index=True)
n = len(df)

df["situacion_final"] = "P"
idx = np.where(df.riesgo)[0]
y = rng.random(len(idx)) < 0.35
df.loc[idx[y], "situacion_final"] = "Y"
df.loc[idx[~y], "situacion_final"] = "R"

a = np.where(df.riesgo, rng.beta(5, 1.6, n), rng.beta(9, 1.2, n)) * 100
df["asistencia_pct"] = np.clip(a, 20, 100).round(1)
x = df.asistencia_pct / 100
p = 1.8 + 5.0 * x**2 - np.where(df.riesgo, 0.5, 0) + rng.normal(0, 0.45, n)
df["promedio_general"] = np.clip(p, 1.0, 7.0).round(1)
df["nivel"] = rng.integers(1, 5, n)
sobre = np.where(df.riesgo, rng.choice([0, 1, 2, 3], n, p=[.35, .35, .2, .1]),
                            rng.choice([0, 1, 2], n, p=[.85, .12, .03]))
df["edad"] = 13 + df.nivel + sobre
df["distancia_km"] = np.clip(rng.lognormal(np.log(2.5), 1.0, n), 0.1, 60).round(2)

retirados = df.index[df.situacion_final == "Y"]
disfrazados = df.loc[retirados].sort_values("asistencia_pct").index[:720]
df.loc[disfrazados, "promedio_general"] = 0.0

# rbd ficticio por comuna-dependencia (2-6 por celda, únicos)
rbd_pool = iter(rng.permutation(np.arange(40000, 50000)))
df["rbd"] = ""
for (cod, dep), grp in df.groupby(["cod_comuna", "dependencia"]):
    k = rng.integers(2, 7)
    codigos = [str(next(rbd_pool)) for _ in range(k)]
    df.loc[grp.index, "rbd"] = rng.choice(codigos, size=len(grp))

df = df.sample(frac=1, random_state=2026).reset_index(drop=True)
df["id_estudiante"] = [f"E{i + 1:05d}" for i in range(len(df))]

COLUMNAS = ["id_estudiante", "rbd", "cod_comuna", "comuna", "dependencia", "nivel", "edad",
            "asistencia_pct", "promedio_general", "distancia_km", "situacion_final"]
df[COLUMNAS].to_csv(CARPETA / "estudiantes_media.csv", index=False)

print("shape:", df[COLUMNAS].shape)
print("Escrito:", (CARPETA / "estudiantes_media.csv").resolve())

---
## Ejercicio 1 — Una variable a la vez (≈20 min)

`estudiantes_media.csv` trae el promedio general y la distancia al establecimiento de 24.000 estudiantes. Antes de cruzar nada, hay que mirar cada variable por separado: ¿qué rango tiene?, ¿hay valores que no deberían estar ahí?, ¿la escala lineal es la correcta?

Ver `lab07-guia-eda.md`, §1–3.

### 1.1 Primero, el error
`df.describe()` y la media de `promedio_general` — en una escala de 1,0 a 7,0, el mínimo sale 0,0. Y el histograma de `distancia_km` con bins por defecto queda todo apretado a la izquierda.

In [ ]:
df = pd.read_csv(CARPETA / "estudiantes_media.csv", dtype={"cod_comuna": str, "rbd": str})
print(df.describe())
print("\npromedio_general medio:", df.promedio_general.mean())

fig, ax = plt.subplots()
ax.hist(df.distancia_km, bins=10)
ax.set_title("distancia_km, bins por defecto")

### 1.2 ✏️ TU TURNO
Construyan, a partir de `df`:

1. `n_disfrazados` = cantidad de filas con `promedio_general == 0.0`.
2. `df1` = copia de `df` con esos valores convertidos a `NaN`. Mismas filas y columnas; el CSV no se toca.
3. `situacion_nulos` = `value_counts()` de `situacion_final` entre las filas con `NaN` en `promedio_general`.
4. `fig1, ax1` = histograma de `distancia_km` en **escala log** (`bins=np.logspace(-1, 2, 40)` + `ax1.set_xscale("log")`), con la mediana marcada (`axvline`).
5. `asimetria_raw` y `asimetria_log` = `Series.skew()` de `distancia_km` y de `np.log10(distancia_km)`.

Celda markdown aparte: ¿qué significa que casi todos los `NaN` de `promedio_general` sean de estudiantes retirados (`Y`)? ¿Se imputan con la media o no?

Pistas: `df1.loc[..., "promedio_general"] = np.nan` · `np.logspace(-1, 2, 40)` cubre de 0,1 a 100 km · `ax1.axvline(df1.distancia_km.median(), ...)`.

In [ ]:
# ✏️ TU TURNO — completen
# n_disfrazados = int((df.promedio_general == 0.0).sum())
# df1 = df.copy()
# df1.loc[df1.promedio_general == 0.0, "promedio_general"] = np.nan
# situacion_nulos = df1.loc[df1.promedio_general.isna(), "situacion_final"].value_counts()
#
# fig1, ax1 = plt.subplots()
# ax1.hist(df1.distancia_km, bins=np.logspace(-1, 2, 40))
# ax1.set_xscale("log")
# ax1.axvline(df1.distancia_km.median(), color="black", linestyle="--", label="mediana")
# ax1.set_xlabel("distancia_km (log)"); ax1.set_ylabel("n estudiantes"); ax1.legend()
#
# asimetria_raw = df1.distancia_km.skew()
# asimetria_log = np.log10(df1.distancia_km).skew()

*(Respondan acá: ¿qué significa que casi todos los `NaN` sean `Y`? ¿Se imputan o no?)*

### ✅ Verificación automática
Ejecuten esta celda cuando tengan `n_disfrazados`, `df1`, `situacion_nulos`, `ax1`, `asimetria_raw` y `asimetria_log`. Debe imprimir 6 ✓.

In [ ]:
def verificar_ej1():
    checks = [
        ("df1 tiene 24.000 filas y 11 columnas",
         lambda: df1.shape == (24000, 11)),
        ("n_disfrazados == 720",
         lambda: n_disfrazados == 720),
        ("promedio_general sin valores fuera de [1, 7] y 720 NaN",
         lambda: df1.promedio_general.dropna().between(1, 7).all()
                 and int(df1.promedio_general.isna().sum()) == 720),
        ("situacion_nulos: la mayoría son retirados (Y)",
         lambda: situacion_nulos.idxmax() == "Y"),
        ("ax1 en escala log con al menos 10 barras",
         lambda: ax1.get_xscale() == "log" and len(ax1.patches) >= 10),
        ("asimetria_raw > 2 y |asimetria_log| < 1",
         lambda: asimetria_raw > 2 and abs(asimetria_log) < 1),
    ]
    return ejecutar_checks(checks)

verificar_ej1()

<details>
<summary><b>Solución propuesta (Ejercicio 1)</b> — clic para desplegar</summary>

```python
n_disfrazados = int((df.promedio_general == 0.0).sum())
df1 = df.copy()
df1.loc[df1.promedio_general == 0.0, "promedio_general"] = np.nan
situacion_nulos = df1.loc[df1.promedio_general.isna(), "situacion_final"].value_counts()

fig1, ax1 = plt.subplots()
ax1.hist(df1.distancia_km, bins=np.logspace(-1, 2, 40))
ax1.set_xscale("log")
ax1.axvline(df1.distancia_km.median(), color="black", linestyle="--", label="mediana")
ax1.set_xlabel("distancia_km (log)"); ax1.set_ylabel("n estudiantes"); ax1.legend()
# variante seaborn: sns.histplot(df1, x="distancia_km", log_scale=True)

asimetria_raw = df1.distancia_km.skew()
asimetria_log = np.log10(df1.distancia_km).skew()
verificar_ej1()
```

El `NaN` de `promedio_general` **no es aleatorio**: 720 de ~957 retirados no tienen nota, porque se retiraron sin rendir el semestre. Imputar con la media o la mediana inventaría un desempeño que nunca se midió; lo correcto es dejarlo como `NaN` y decidir en el registro de decisiones si esas filas se excluyen del análisis de rendimiento.
</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
_n_disfrazados = int((df.promedio_general == 0.0).sum())
df_sol1 = df.copy()
df_sol1.loc[df_sol1.promedio_general == 0.0, "promedio_general"] = np.nan
_situacion_nulos = df_sol1.loc[df_sol1.promedio_general.isna(), "situacion_final"].value_counts()

_fig1, _ax1 = plt.subplots()
_ax1.hist(df_sol1.distancia_km, bins=np.logspace(-1, 2, 40))
_ax1.set_xscale("log")
_ax1.axvline(df_sol1.distancia_km.median(), color="black", linestyle="--")

_asimetria_raw = df_sol1.distancia_km.skew()
_asimetria_log = np.log10(df_sol1.distancia_km).skew()

assert _n_disfrazados == 720
assert df_sol1.shape == (24000, 11)
assert int(df_sol1.promedio_general.isna().sum()) == 720
assert _situacion_nulos.idxmax() == "Y"
assert _asimetria_raw > 2 and abs(_asimetria_log) < 1, "La solución de referencia del Ej. 1 no pasa la verificación"
print("n_disfrazados:", _n_disfrazados, "· asimetria_raw:", round(_asimetria_raw, 2),
      "· asimetria_log:", round(_asimetria_log, 3))
_situacion_nulos

---
## Ejercicio 2 — Relaciones y correlación (≈20 min)

¿Asistir más se relaciona con mejor promedio? Con 24.000 puntos un `scatter` normal es una mancha, y los 720 ceros disfrazados de nulo alteran la correlación sin que se note a simple vista.

Ver `lab07-guia-eda.md`, §5–6.

### 2.1 Primero, el error
`plt.scatter` de asistencia vs. promedio con las 24.000 filas **de `df` crudo** (con los ceros adentro).

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df.asistencia_pct, df.promedio_general, s=4)
ax.set_title("Asistencia vs. promedio (crudo, 24.000 puntos)")

### 2.2 ✏️ TU TURNO
Sobre `df_sol1` (o su `df1`):

1. `fig2, ax2` = versión legible: `ax2.hexbin(x, y, gridsize=40, mincnt=1)` **o** `ax2.scatter` sobre `df_sol1.sample(5000, random_state=7)` con `alpha <= 0.3`. Título y ejes con unidades.
2. `corr` = diccionario con `pearson_crudo`, `spearman_crudo` (sobre `df`, con los ceros) y `pearson_limpio`, `spearman_limpio` (sobre `df_sol1`, sin los ceros disfrazados), usando el helper `spearman(a, b)` del Setup.
3. Celda markdown: ¿por qué Pearson cambió tanto entre crudo y limpio, y Spearman casi nada?
4. `fig2b, ax2b` = boxplot de `edad` por `situacion_final`, en el orden `["P", "R", "Y"]`.

Pistas: `dropna()` antes de correlacionar con `.corr()` normal (Pearson ignora NaN por pares, así que pueden pasar `df_sol1.asistencia_pct` y `df_sol1.promedio_general` directo) · `ax2b.boxplot([...], tick_labels=["P", "R", "Y"])`.

In [ ]:
# ✏️ TU TURNO — completen
# fig2, ax2 = plt.subplots()
# hb = ax2.hexbin(df_sol1.asistencia_pct, df_sol1.promedio_general, gridsize=40, mincnt=1)
# ax2.set_xlabel("asistencia (%)"); ax2.set_ylabel("promedio general (1-7)")
# ax2.set_title("Asistencia vs. promedio, sin los ceros disfrazados")
#
# corr = {
#     "pearson_crudo": df.asistencia_pct.corr(df.promedio_general),
#     "spearman_crudo": spearman(df.asistencia_pct, df.promedio_general),
#     "pearson_limpio": df_sol1.asistencia_pct.corr(df_sol1.promedio_general),
#     "spearman_limpio": spearman(df_sol1.asistencia_pct, df_sol1.promedio_general),
# }
#
# fig2b, ax2b = plt.subplots()
# grupos = [df_sol1.loc[df_sol1.situacion_final == s, "edad"] for s in ["P", "R", "Y"]]
# ax2b.boxplot(grupos, tick_labels=["P", "R", "Y"])
# ax2b.set_xlabel("situación final"); ax2b.set_ylabel("edad (años)")

*(Respondan acá: ¿por qué Pearson cambió tanto y Spearman casi nada?)*

### ✅ Verificación automática

In [ ]:
_corr_ref = {
    "pearson_crudo": df.asistencia_pct.corr(df.promedio_general),
    "spearman_crudo": spearman(df.asistencia_pct, df.promedio_general),
    "pearson_limpio": df_sol1.asistencia_pct.corr(df_sol1.promedio_general),
    "spearman_limpio": spearman(df_sol1.asistencia_pct, df_sol1.promedio_general),
}


def _ax2_es_legible():
    import matplotlib.collections as mc
    if any(isinstance(c, mc.PolyCollection) for c in ax2.collections):
        return True
    if ax2.collections:
        pts = ax2.collections[0]
        n_pts = pts.get_offsets().shape[0]
        alpha = pts.get_alpha()
        return n_pts <= 5000 and alpha is not None and alpha <= 0.3
    return False


def verificar_ej2():
    checks = [
        ("ax2 es legible (hexbin, o scatter con ≤ 5.000 puntos y alpha ≤ 0,3)",
         _ax2_es_legible),
        ("Las 4 correlaciones coinciden con la referencia (±0,01)",
         lambda: all(abs(corr[k] - _corr_ref[k]) < 0.01 for k in _corr_ref)),
        ("Pearson sube > 0,05 al limpiar y Spearman casi no cambia (< 0,02)",
         lambda: corr["pearson_limpio"] - corr["pearson_crudo"] > 0.05
                 and abs(corr["spearman_limpio"] - corr["spearman_crudo"]) < 0.02),
        ("ax2b tiene 3 categorías",
         lambda: len(ax2b.get_xticklabels()) == 3),
    ]
    return ejecutar_checks(checks)

verificar_ej2()

<details>
<summary><b>Solución propuesta (Ejercicio 2)</b> — clic para desplegar</summary>

```python
fig2, ax2 = plt.subplots()
hb = ax2.hexbin(df_sol1.asistencia_pct, df_sol1.promedio_general, gridsize=40, mincnt=1)
ax2.set_xlabel("asistencia (%)"); ax2.set_ylabel("promedio general (1-7)")
ax2.set_title("Asistencia vs. promedio, sin los ceros disfrazados")
# variante seaborn: sns.regplot(df_sol1.sample(5000, random_state=7), x="asistencia_pct", y="promedio_general",
#                                scatter_kws={"alpha": 0.3})

corr = {
    "pearson_crudo": df.asistencia_pct.corr(df.promedio_general),
    "spearman_crudo": spearman(df.asistencia_pct, df.promedio_general),
    "pearson_limpio": df_sol1.asistencia_pct.corr(df_sol1.promedio_general),
    "spearman_limpio": spearman(df_sol1.asistencia_pct, df_sol1.promedio_general),
}

fig2b, ax2b = plt.subplots()
grupos = [df_sol1.loc[df_sol1.situacion_final == s, "edad"] for s in ["P", "R", "Y"]]
ax2b.boxplot(grupos, tick_labels=["P", "R", "Y"])
ax2b.set_xlabel("situación final"); ax2b.set_ylabel("edad (años)")
verificar_ej2()
```

Pearson mide relación **lineal** y es sensible a outliers: los 720 ceros disfrazados son puntos lejos de la nube (asistencia baja, promedio "0") que tiran la recta hacia abajo. Spearman compara **rangos**, y un cero sigue siendo el valor más bajo del ranking con o sin disfraz — por eso casi no se mueve.
</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
corr = dict(_corr_ref)
fig2, ax2 = plt.subplots()
ax2.hexbin(df_sol1.asistencia_pct, df_sol1.promedio_general, gridsize=40, mincnt=1)
ax2.set_xlabel("asistencia (%)"); ax2.set_ylabel("promedio general (1-7)")

fig2b, ax2b = plt.subplots()
_grupos = [df_sol1.loc[df_sol1.situacion_final == s, "edad"] for s in ["P", "R", "Y"]]
ax2b.boxplot(_grupos, tick_labels=["P", "R", "Y"])

assert verificar_ej2(), "La solución de referencia del Ej. 2 no pasa la verificación"
pd.Series(corr)

---
## Ejercicio 3 — Paradoja de Simpson (≈15 min)

Penco y Tomé son las dos comunas con más establecimientos públicos de la provincia. Comparadas en total, Penco parece "mejor". Comparadas dependencia por dependencia, es al revés.

Ver `lab07-guia-eda.md`, §8.

### 3.1 Primero, el error
Tasa de riesgo (`situacion_final in {"R", "Y"}`) por comuna, **solo el total**, para Penco y Tomé.

In [ ]:
riesgo_total = df.assign(riesgo=df.situacion_final.isin(["R", "Y"])).groupby("comuna").riesgo.mean()
print(riesgo_total[["Penco", "Tomé"]])
print("\n¿Penco es 'mejor' que Tomé en todo?")

### 3.2 ✏️ TU TURNO

1. `tab3` = tabla comuna × dependencia (índice `["Penco", "Tomé"]`, columnas = las 3 dependencias) con la tasa de riesgo de cada celda.
2. `comp3` = `pd.crosstab(df.comuna, df.dependencia, normalize="index")`: la composición por dependencia de cada comuna.
3. `fig3, ax3` = barras agrupadas: tasa de riesgo por dependencia, una barra por comuna.
4. `conclusion3` = `"A"`, `"B"` o `"C"`:
   - A) Los establecimientos de Penco son mejores en todas las dependencias.
   - B) Penco tiene menor tasa total porque su matrícula se concentra en dependencias de bajo riesgo; dentro de cada dependencia su tasa es *mayor* que la de Tomé.
   - C) La diferencia total es ruido: con ~2.000 estudiantes por comuna no se puede decir nada.

Pistas: `groupby(["comuna", "dependencia"]).riesgo.mean().unstack()` · `ax3.bar(x - width/2, ...)` y `ax3.bar(x + width/2, ...)` para las barras agrupadas.

In [ ]:
# ✏️ TU TURNO — completen
# _con_riesgo = df.assign(riesgo=df.situacion_final.isin(["R", "Y"]))
# tab3 = _con_riesgo.groupby(["comuna", "dependencia"]).riesgo.mean().unstack().loc[["Penco", "Tomé"]]
# comp3 = pd.crosstab(df.comuna, df.dependencia, normalize="index")
#
# fig3, ax3 = plt.subplots()
# x = np.arange(len(tab3.columns)); width = 0.35
# ax3.bar(x - width / 2, tab3.loc["Penco"], width, label="Penco")
# ax3.bar(x + width / 2, tab3.loc["Tomé"], width, label="Tomé")
# ax3.set_xticks(x); ax3.set_xticklabels(tab3.columns)
# ax3.set_ylabel("tasa de riesgo"); ax3.legend()
#
# conclusion3 = "B"

### ✅ Verificación automática

In [ ]:
_con_riesgo_ref = df.assign(riesgo=df.situacion_final.isin(["R", "Y"]))
_tab3_ref = _con_riesgo_ref.groupby(["comuna", "dependencia"]).riesgo.mean().unstack().loc[["Penco", "Tomé"]]


def verificar_ej3():
    checks = [
        ("tab3 tiene forma 2 x 3 e igual a la referencia (±1e-9)",
         lambda: tab3.shape == (2, 3) and np.allclose(tab3.loc[["Penco", "Tomé"], _tab3_ref.columns],
                                                        _tab3_ref, atol=1e-9)),
        ("Penco > Tomé en las 3 dependencias",
         lambda: (tab3.loc["Penco"] > tab3.loc["Tomé"]).all()),
        ("comp3 suma 1 por fila",
         lambda: np.allclose(comp3.sum(axis=1), 1.0)),
        ("ax3 tiene al menos 6 barras",
         lambda: len(ax3.patches) >= 6),
        ('conclusion3 == "B"',
         lambda: conclusion3 == "B"),
    ]
    return ejecutar_checks(checks)

verificar_ej3()

<details>
<summary><b>Solución propuesta (Ejercicio 3)</b> — clic para desplegar</summary>

```python
_con_riesgo = df.assign(riesgo=df.situacion_final.isin(["R", "Y"]))
tab3 = _con_riesgo.groupby(["comuna", "dependencia"]).riesgo.mean().unstack().loc[["Penco", "Tomé"]]
comp3 = pd.crosstab(df.comuna, df.dependencia, normalize="index")

fig3, ax3 = plt.subplots()
x = np.arange(len(tab3.columns)); width = 0.35
ax3.bar(x - width / 2, tab3.loc["Penco"], width, label="Penco")
ax3.bar(x + width / 2, tab3.loc["Tomé"], width, label="Tomé")
ax3.set_xticks(x); ax3.set_xticklabels(tab3.columns)
ax3.set_ylabel("tasa de riesgo"); ax3.set_title("Riesgo por dependencia: Penco siempre peor")
ax3.legend()
# variante seaborn: sns.barplot(_con_riesgo[_con_riesgo.comuna.isin(["Penco","Tomé"])],
#                                x="dependencia", y="riesgo", hue="comuna", estimator="mean")

conclusion3 = "B"
verificar_ej3()
```

La tasa total de Penco (9,5 %) es más baja que la de Tomé (15,4 %) solo porque Penco tiene más matrícula particular pagada (bajo riesgo) y Tomé tiene mucha matrícula pública (que en ambas comunas es la de mayor riesgo). Dentro de cada dependencia Penco es siempre peor. Es la paradoja de Simpson: el **confusor** es la composición por dependencia, y agregar sin segmentar oculta la dirección real del efecto.
</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
tab3 = _tab3_ref
comp3 = pd.crosstab(df.comuna, df.dependencia, normalize="index")
fig3, ax3 = plt.subplots()
_x = np.arange(len(tab3.columns)); _width = 0.35
ax3.bar(_x - _width / 2, tab3.loc["Penco"], _width, label="Penco")
ax3.bar(_x + _width / 2, tab3.loc["Tomé"], _width, label="Tomé")
conclusion3 = "B"

assert verificar_ej3(), "La solución de referencia del Ej. 3 no pasa la verificación"
df_sol3 = tab3
df_sol3

---
## Ejercicio 4 — Una figura que comunica (≈10 min, opcional; puede quedar para la casa)

Rehagan una figura de los ejercicios anteriores (a elección) para que vaya a un informe.

Ver `lab07-guia-eda.md`, §9.

### ✏️ TU TURNO
1. `fig4, ax4` = la figura elegida, con:
   - Título = el **hallazgo** en palabras (no el nombre de una columna), de al menos 25 caracteres.
   - Ejes con unidades entre paréntesis (al menos uno).
   - Escala declarada si usan log.
2. `guardar_figura(fig4, "ej4.png")` (guarda en `figs/`, la crea si no existe).

Pistas: `ax4.set_title("Penco tiene mayor riesgo en cada dependencia")` en vez de `ax4.set_title("tasa de riesgo")`.

In [ ]:
# ✏️ TU TURNO — completen
# fig4, ax4 = plt.subplots()
# x = np.arange(len(tab3.columns)); width = 0.35
# ax4.bar(x - width / 2, tab3.loc["Penco"], width, label="Penco")
# ax4.bar(x + width / 2, tab3.loc["Tomé"], width, label="Tomé")
# ax4.set_xticks(x); ax4.set_xticklabels(tab3.columns)
# ax4.set_title("Penco es peor que Tomé en las tres dependencias")
# ax4.set_xlabel("dependencia"); ax4.set_ylabel("tasa de riesgo (%)"); ax4.legend()
# guardar_figura(fig4, "ej4.png")

### ✅ Verificación automática

In [ ]:
_NOMBRES_COLUMNA = set(df.columns) | {"tasa de riesgo", "distancia_km", "promedio_general", "asistencia_pct"}


def verificar_ej4():
    checks = [
        ("Existe figs/ej4.png",
         lambda: (FIGS / "ej4.png").exists()),
        ("Título de ≥ 25 caracteres y distinto de un nombre de columna",
         lambda: len(ax4.get_title()) >= 25 and ax4.get_title() not in _NOMBRES_COLUMNA),
        ("xlabel e ylabel no vacíos y al menos uno con unidades entre paréntesis",
         lambda: bool(ax4.get_xlabel()) and bool(ax4.get_ylabel())
                 and ("(" in ax4.get_xlabel() or "(" in ax4.get_ylabel())),
        ("La figura tiene un solo Axes o un suptitle",
         lambda: len(fig4.axes) == 1 or bool(fig4._suptitle)),
    ]
    return ejecutar_checks(checks)

verificar_ej4()

<details>
<summary><b>Solución propuesta (Ejercicio 4)</b> — clic para desplegar</summary>

```python
fig4, ax4 = plt.subplots()
x = np.arange(len(tab3.columns)); width = 0.35
ax4.bar(x - width / 2, tab3.loc["Penco"] * 100, width, label="Penco")
ax4.bar(x + width / 2, tab3.loc["Tomé"] * 100, width, label="Tomé")
ax4.set_xticks(x); ax4.set_xticklabels(tab3.columns)
ax4.set_title("Penco es peor que Tomé en las tres dependencias")
ax4.set_xlabel("dependencia"); ax4.set_ylabel("tasa de riesgo (%)"); ax4.legend()
guardar_figura(fig4, "ej4.png")
verificar_ej4()
```
</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
fig4, ax4 = plt.subplots()
_x4 = np.arange(len(tab3.columns)); _w4 = 0.35
ax4.bar(_x4 - _w4 / 2, tab3.loc["Penco"] * 100, _w4, label="Penco")
ax4.bar(_x4 + _w4 / 2, tab3.loc["Tomé"] * 100, _w4, label="Tomé")
ax4.set_xticks(_x4); ax4.set_xticklabels(tab3.columns)
ax4.set_title("Penco es peor que Tomé en las tres dependencias")
ax4.set_xlabel("dependencia"); ax4.set_ylabel("tasa de riesgo (%)"); ax4.legend()
guardar_figura(fig4, "ej4.png")
assert verificar_ej4(), "La solución de referencia del Ej. 4 no pasa la verificación"

---
## 5 · Ahora con SU proyecto — plantilla de `03-eda.ipynb` (11:00–11:35)

Lo que sigue no es un ejercicio: es el **esqueleto** del tercer notebook de su repositorio. Cópienlo a `notebooks/03-eda.ipynb` y complétenlo con el EDA real de su proyecto.

In [ ]:
# ==========================================================
# PLANTILLA 03-eda — completar con SU proyecto
# ==========================================================
from pathlib import Path
RAIZ = Path("..")                        # notebooks/ → raíz del repo
PROCESSED = RAIZ / "data" / "processed"  # 03-eda lee SOLO desde aquí
FIGS = RAIZ / "figs"; FIGS.mkdir(exist_ok=True)


def guardar_figura(fig, nombre):
    fig.savefig(FIGS / nombre, dpi=150, bbox_inches="tight")


# 0. Qué es una fila (unidad de análisis) y cuántas hay
# 1. Variable objetivo: n con target, balance de clases, nulos
# 2. Univariado: solo las variables que aparecen en su pregunta de datos
# 3. Bivariado contra la variable objetivo
# 4. Segmentación: ¿el hallazgo se mantiene con hue=/col= por la variable de composición?

BITACORA_EDA = [
    {"hallazgo": "",           # frase con número: "El 12 % de ... "
     "figura": "",             # figs/xx.png
     "celda": "",              # "03-eda.ipynb, celda N"
     "pregunta": "",           # qué término de la pregunta de datos toca
     "decision_o_hipotesis": ""},
    # ... mínimo 3 filas hoy
]
pd.DataFrame(BITACORA_EDA)

Reglas para la bitácora: **el título de cada figura es el hallazgo**, con unidades; ninguna cifra sin la celda que la produce; nada de `pairplot` sobre datasets grandes; muestreo con semilla fija cuando el dataset no cabe cómodo en una figura.

---
### Checklist de salida de la Clase 7

- [ ] Ejercicios 1, 2 y 3 pasan su verificación automática (el 4 puede quedar para la casa)
- [ ] `notebooks/03-eda.ipynb` creado en el repo, leyendo solo desde `data/processed/`
- [ ] Unidad de análisis y variable objetivo descritas (n, balance, nulos)
- [ ] 3 filas de la bitácora de EDA, cada una con figura en `figs/` y su celda
- [ ] Gate del Informe 2 probado: su repo clonado en una carpeta vacía, `01` y `02` corren de punta a punta
- [ ] Primera falla del gate (si hubo) anotada y con plan de arreglo antes del domingo